# Bias–variance tradeoff: interactive demo

We fit polynomials (Chebyshev basis, optional ridge penalty) to noisy samples of a target function $f(x)$ on $[-1,1]$.
For each setting we draw $M=60$ independent training sets and estimate, on a dense test grid,

$$\mathbb{E}\big[(y-\hat f(x))^2\big]=\underbrace{(\bar f(x)-f(x))^2}_{\text{bias}^2}+\underbrace{\mathrm{Var}[\hat f(x)]}_{\text{variance}}+\underbrace{\sigma^2}_{\text{noise}},\qquad \bar f=\mathbb{E}_{\mathcal D}[\hat f].$$

Run the cells, then play with the sliders in the last one. Requires `numpy`, `matplotlib`, `ipywidgets`.


| Model complexity | Bias         | Variance | Typical behavior    |
| ---------------- | ------------ | -------- | ------------------- |
| Very low         | High         | Low      | Underfitting        |
| Moderate         | Moderate/low | Moderate | Good generalization |
| Very high        | Low          | High     | Overfitting         |


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as W
from numpy.polynomial import chebyshev as C
from IPython.display import display

M, NMAX, DMAX, NG, SHOW = 60, 100, 15, 200, 40
grid = np.linspace(-1, 1, NG)
GB = C.chebvander(grid, DMAX)                      # (NG, DMAX+1)

TARGETS = {
    "sine":          lambda u: np.sin(np.pi * u),
    "bump (Runge)":  lambda u: 1 / (1 + 25 * u**2),
    "sharp step":    lambda u: np.tanh(6 * u),
    "cubic":         lambda u: 1.5 * (u**3 - 0.6 * u),
}

def make_data(seed):
    """Fixed uniform x's and N(0,1) noise, so sliders change results smoothly."""
    r = np.random.default_rng(seed)
    return r.uniform(-1, 1, (M, NMAX)), r.standard_normal((M, NMAX))

def target(x, name, freq):
    return TARGETS[name](freq * x)

def fit_predict(X, Y, K, lam):
    """Batched ridge fit of degree K. X, Y: (M, n). Returns predictions on grid, (M, NG)."""
    Phi = C.chebvander(X, K)                           # (M, n, K+1) via broadcasting
    A = Phi.transpose(0, 2, 1) @ Phi + (lam + 1e-8) * np.eye(K + 1)
    b = Phi.transpose(0, 2, 1) @ Y[..., None]
    w = np.linalg.solve(A, b)[..., 0]                  # (M, K+1)
    return w @ GB[:, :K + 1].T

def sweep(xs, zs, n, sigma, lam, freq, name):
    ft = target(grid, name, freq)
    X = xs[:, :n]
    Y = target(X, name, freq) + sigma * zs[:, :n]
    res = []
    for K in range(DMAX + 1):
        P = fit_predict(X, Y, K, lam)
        mean = P.mean(0)
        res.append(dict(P=P, mean=mean,
                        bias2=np.mean((mean - ft) ** 2),
                        var=np.mean(P.var(0))))
    return ft, res, X, Y


In [ ]:
BL, OR = "#2a78d6", "#eb6834"
_cache = {}

def cached_sweep(seed, n, sigma, lam, freq, name):
    key = (seed, n, sigma, lam, freq, name)
    if key not in _cache:
        _cache.clear()
        _cache[key] = sweep(*DATA[seed], n, sigma, lam, freq, name)
    return _cache[key]

DATA = {}
def plot(deg, sigma, n, loglam, freq, name, seed):
    if seed not in DATA:
        DATA[seed] = make_data(seed)
    lam = 0.0 if loglam <= -7 else 10.0 ** loglam
    ft, res, X, Y = cached_sweep(seed, n, sigma, lam, freq, name)
    R = res[deg]
    total = np.array([r["bias2"] + r["var"] + sigma**2 for r in res])

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2))
    for p in R["P"][:SHOW]:
        a1.plot(grid, p, color=OR, alpha=0.2, lw=1)
    a1.plot(grid, ft, "k", lw=2, label="true function")
    a1.plot(grid, R["mean"], color=BL, lw=2.5, label="average fit")
    a1.plot([], [], color=OR, alpha=0.6, label=f"fits on {SHOW} datasets")
    a1.scatter(X[0], Y[0], s=14, c="gray", alpha=0.8, zorder=3, label="one training set")
    a1.set(xlim=(-1, 1), ylim=(-2, 2), xlabel="x", ylabel="y", title=f"Degree {deg} fits")
    a1.legend(loc="upper right", fontsize=8, framealpha=0.9)
    a1.grid(alpha=0.25)

    ks = np.arange(DMAX + 1)
    a2.plot(ks, [r["bias2"] for r in res], color=BL, lw=2, label="bias$^2$")
    a2.plot(ks, [r["var"] for r in res], color=OR, lw=2, label="variance")
    a2.plot(ks, total, "k", lw=2.5, label="bias$^2$ + variance + $\\sigma^2$")
    a2.axhline(sigma**2, color="gray", ls="--", lw=1.2, label="noise $\\sigma^2$")
    a2.axvline(deg, color="gray", ls=":", lw=1)
    a2.plot(deg, min(total[deg], 3 * total.min()), "ko", ms=7)
    best = int(total.argmin())
    a2.plot(best, total[best], "o", mfc="none", mec="k", ms=14, mew=1.5)
    a2.set(xlim=(0, DMAX), ylim=(0, max(3 * total.min(), 0.05)),
           xlabel="polynomial degree", ylabel="error", title="Error decomposition (ring = best degree)")
    a2.legend(loc="upper center", fontsize=8, framealpha=0.9)
    a2.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

    print(f"degree {deg}:  bias² = {R['bias2']:.4f}   variance = {R['var']:.4f}   "
          f"noise σ² = {sigma**2:.4f}   expected test error = {total[deg]:.4f}")


In [ ]:
style = {"description_width": "110px"}
lay = W.Layout(width="420px")
kw = dict(style=style, layout=lay, continuous_update=False)

w_deg   = W.IntSlider(value=3, min=0, max=DMAX, step=1, description="Complexity", style=style, layout=lay, continuous_update=True)
w_sigma = W.FloatSlider(value=0.3, min=0, max=1, step=0.05, description="Noise σ", **kw)
w_n     = W.IntSlider(value=25, min=8, max=NMAX, step=1, description="Training size", **kw)
w_lam   = W.FloatSlider(value=-7, min=-7, max=1, step=0.5, description="log10 ridge λ", readout_format=".1f", **kw)
w_freq  = W.FloatSlider(value=1.0, min=0.5, max=3.0, step=0.1, description="Frequency", **kw)
w_fn    = W.Dropdown(options=list(TARGETS), value="sine", description="Target", style=style, layout=lay)
w_seed  = W.IntText(value=1, description="Data seed", style=style, layout=W.Layout(width="200px"))
b_new   = W.Button(description="Resample data", icon="refresh")
b_new.on_click(lambda _: setattr(w_seed, "value", w_seed.value + 1))

ui = W.VBox([
    W.HBox([W.VBox([w_deg, w_sigma, w_n]), W.VBox([w_lam, w_freq, w_fn])]),
    W.HBox([w_seed, b_new]),
])
out = W.interactive_output(plot, dict(deg=w_deg, sigma=w_sigma, n=w_n, loglam=w_lam,
                                      freq=w_freq, name=w_fn, seed=w_seed))
display(ui, out)


Output()

**Things to try**

* Complexity 1: the orange fits bunch together but all miss the target → high bias, low variance.
* Complexity 15 with training size 10–15: fits scatter wildly → high variance.
* Same overfit setting with log λ ≈ −2: ridge cuts the variance at the cost of a bit of bias.
* Raise noise σ: the optimal degree drops. Raise training size: it rises.
* Frequency 2.5, noise 0.1, size 100: more complexity is needed before the bias collapses.
